# Lezione 4 — Da comprensione a preparazione: pulisci e comprendi i tuoi dati

**Durata stimata: 120 minuti**

## Obiettivi di apprendimento

- Applicare statistica descrittiva (univariate, correlazioni, istogrammi) per capire un dataset.
- Rilevare e decidere cosa fare con valori assenti o codificati male.
- Pulire e trasformare un dataset reale con pandas (nomi delle colonne, valori incoerenti, categorie poco frequenti).
- Spiegare cos'è il feature engineering e perché conta.

Nella Lezione 3 hai caricato il dataset di 57.691 ricette di cucina nel DataFrame `recipes`. In questa lezione farai due cose con esso: prima, **capirlo** (Data Understanding: questi dati rappresentano bene il problema di identificare la cucina di una ricetta a partire dai suoi ingredienti?), e poi, **pulirlo e trasformarlo** (Data Preparation) per lasciarlo pronto da modellare nella prossima lezione.

In parallelo, vedrai come queste stesse due fasi si sono applicate al caso reale dell'assicurazione sanitaria e dei pazienti con insufficienza cardiaca congestizia (CHF) — quel caso non ha codice perché i dati ospedalieri non sono pubblici, ma la logica è identica a quella che applicherai qui con codice reale.

## 0. Promemoria: caricare il dataset

Questo l'hai già fatto nella Lezione 3. Lo ripetiamo qui affinché questo notebook sia autocontenuto e tu possa eseguirlo dall'inizio alla fine senza dipendere da un altro notebook.

In [ ]:
import pandas as pd  # libreria standard per lavorare con dati in tabelle (DataFrame)
pd.set_option('display.max_columns', None)  # mostra tutte le colonne quando stampa
import numpy as np  # libreria per operazioni numeriche e array
import re  # libreria per espressioni regolari (ricerca di pattern nel testo)

recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

print("Data read into dataframe!")

In [ ]:
recipes.head()  # prime 5 righe, per ricordare la forma del dataset

In [ ]:
recipes.shape  # (righe, colonne) -> (57691, 385): 57.691 ricette, 384 ingredienti + 1 colonna della cucina

## 1. Data Understanding: statistica descrittiva

La domanda di questa fase è: **questi dati rappresentano bene il problema che vuoi risolvere?**

I tre strumenti classici sono:

1. **Univariate**: media, mediana, minimo, massimo, deviazione standard di ogni variabile.
2. **Correlazioni a coppie**: per rilevare variabili ridondanti.
3. **Istogrammi**: per vedere distribuzioni e decidere come consolidare categorie.

In un dataset come `recipes`, dove quasi tutte le colonne di ingredienti sono binarie (0/1) una volta pulite, la statistica univariata più utile è semplicemente "in che percentuale di ricette compare questo ingrediente?" — ci arriveremo più avanti in questo stesso notebook (la sezione dei profili per cucina). Prima di questo, iniziamo dal più basilare: verificare che gli ingredienti che ci aspettiamo di trovare esistano davvero come colonne.

### 1.1 Esistono gli ingredienti che ci aspettiamo?

Usiamo le espressioni regolari (`re`) per cercare, dentro i nomi delle colonne, corrispondenze parziali con "rice" (riso), "wasabi" e "soy" (soia).

In [ ]:
ingredients = list(recipes.columns.values)  # lista con tutti i nomi delle colonne

print([match.group(0) for ingredient in ingredients
       for match in [(re.compile(".*(rice).*")).search(ingredient)] if match])

print([match.group(0) for ingredient in ingredients
       for match in [(re.compile(".*(wasabi).*")).search(ingredient)] if match])

print([match.group(0) for ingredient in ingredients
       for match in [(re.compile(".*(soy).*")).search(ingredient)] if match])

Confermiamo che esistono le colonne `rice`, `wasabi` e `soy_sauce`.

**Domanda da conservare:** se una ricetta contiene riso + wasabi + salsa di soia contemporaneamente, potremmo affermare con sicurezza che è cucina giapponese? Metteremo alla prova quella domanda più avanti, con dati reali, dopo aver pulito il dataset.

### 1.2 La sfumatura dei valori assenti

Prima di continuare, un inciso concettuale importante (senza codice, perché `recipes` non ha esattamente questo problema, ma lo incontrerai in dataset reali): quando una variabile ha valori assenti, "assente" può significare cose molto diverse — "no", "zero", "non lo sappiamo" — oppure può nascondere un valore non valido travestito da valido, come un "999" in una colonna di età che in realtà significa "sconosciuto". Se non lo rilevi, il tuo programma lo tratta come un dato numerico legittimo e distorce qualsiasi statistica che calcoli. Per questo il primo passo quando ricevi un nuovo dataset è sempre guardare le sue statistiche univariate (minimo, massimo, media) prima di fidartene ciecamente.

### 1.3 Il caso CHF: quando comprendere i dati ti obbliga a tornare indietro

*(Caso narrativo, senza codice — questi dati ospedalieri non sono disponibili pubblicamente.)*

Nel caso dell'assicurazione sanitaria, il team ha inizialmente definito un'"ammissione per CHF" come qualsiasi ammissione con una **diagnosi principale** di insufficienza cardiaca congestizia. Applicando statistica descrittiva sui dati raccolti, hanno scoperto che quella definizione **non catturava tutte le ammissioni per CHF che l'esperienza clinica si aspettava**. La soluzione non è stata forzare il modello con dati incompleti: il team **è tornato alla fase di Data Collection** e ha ampliato la definizione per includere anche diagnosi secondarie e terziarie di CHF. Questo è l'esempio perfetto del fatto che la metodologia è un ciclo, non una linea retta: più lavori sul problema, più impari, e quell'apprendimento raffina le fasi precedenti.

## 2. Data Preparation: pulire `recipes`

Data Preparation è come lavare verdure appena comprate: togliere la terra e le imperfezioni prima di cucinare. È anche come tagliare una cipolla fine affinché il suo sapore si distribuisca meglio nel sugo — trasformare i dati affinché sia più facile lavorarci.

Un dato importante: insieme alla raccolta e alla comprensione, la preparazione dei dati di solito consuma tra il **70% e il 90%** del tempo totale di un progetto di scienza dei dati. Automatizzare parte di questo processo nel database può abbassarlo fino al 50%.

Iniziamo guardando la variabile della cucina con `.value_counts()`, un metodo di pandas che conta quante volte compare ogni valore distinto di una colonna.

In [ ]:
recipes["country"].value_counts()  # tabella di frequenze della colonna "country"

Questo sguardo scopre quattro problemi di qualità dei dati:

1. La colonna si chiama `country`, ma in realtà contiene il **tipo di cucina** — il nome è sbagliato.
2. I nomi delle cucine mescolano maiuscole e minuscole in modo incoerente.
3. Ci sono cucine duplicate come variazioni del nome del paese (es. "Vietnam" vs "Vietnamese").
4. Alcune cucine hanno pochissime ricette.

Sistemiamo i quattro, uno per uno.

### 2.1 Sistemare il nome della colonna

In [ ]:
column_names = recipes.columns.values  # array con i nomi delle colonne
column_names[0] = "cuisine"              # la prima colonna passa da "country" a "cuisine"
recipes.columns = column_names           # riassegna i nomi corretti

recipes

### 2.2 Minuscolo, affinché "Italian" e "italian" non contino come valori distinti

In [ ]:
recipes["cuisine"] = recipes["cuisine"].str.lower()

### 2.3 Unificare nomi duplicati

`.loc[condizione, colonna] = valore` è il pattern di pandas per: localizzare le righe che soddisfano una condizione, e sovrascrivere il valore di una colonna in quelle righe. Qui lo usiamo riga per paese per lasciare un solo nome di cucina per ciascuno.

In [ ]:
recipes.loc[recipes["cuisine"] == "austria", "cuisine"] = "austrian"
recipes.loc[recipes["cuisine"] == "australia", "cuisine"] = "australian"
recipes.loc[recipes["cuisine"] == "belgium", "cuisine"] = "belgian"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "canada", "cuisine"] = "canadian"
recipes.loc[recipes["cuisine"] == "netherlands", "cuisine"] = "dutch"
recipes.loc[recipes["cuisine"] == "france", "cuisine"] = "french"
recipes.loc[recipes["cuisine"] == "germany", "cuisine"] = "german"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"
recipes.loc[recipes["cuisine"] == "indonesia", "cuisine"] = "indonesian"
recipes.loc[recipes["cuisine"] == "iran", "cuisine"] = "iranian"
recipes.loc[recipes["cuisine"] == "italy", "cuisine"] = "italian"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "israel", "cuisine"] = "jewish"
recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "lebanon", "cuisine"] = "lebanese"
recipes.loc[recipes["cuisine"] == "malaysia", "cuisine"] = "malaysian"
recipes.loc[recipes["cuisine"] == "mexico", "cuisine"] = "mexican"
recipes.loc[recipes["cuisine"] == "pakistan", "cuisine"] = "pakistani"
recipes.loc[recipes["cuisine"] == "philippines", "cuisine"] = "philippine"
recipes.loc[recipes["cuisine"] == "scandinavia", "cuisine"] = "scandinavian"
recipes.loc[recipes["cuisine"] == "spain", "cuisine"] = "spanish_portuguese"
recipes.loc[recipes["cuisine"] == "portugal", "cuisine"] = "spanish_portuguese"
recipes.loc[recipes["cuisine"] == "switzerland", "cuisine"] = "swiss"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "turkey", "cuisine"] = "turkish"
recipes.loc[recipes["cuisine"] == "vietnam", "cuisine"] = "vietnamese"
recipes.loc[recipes["cuisine"] == "uk-and-ireland", "cuisine"] = "uk-and-irish"
recipes.loc[recipes["cuisine"] == "irish", "cuisine"] = "uk-and-irish"

recipes

### 2.4 Eliminare cucine con meno di 50 ricette

Prima ricalcoliamo la tabella delle frequenze già con i nomi unificati, e teniamo le cucine che superano la soglia.

In [ ]:
recipes_counts = recipes["cuisine"].value_counts()
recipes_counts

In [ ]:
cuisines_indices = recipes_counts > 50  # True/False a seconda se la cucina supera 50 ricette
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])

print("Cucine mantenute:")
print(cuisines_to_keep)

`.isin()` è un metodo di pandas che verifica, riga per riga, se il valore di una colonna è dentro una lista data. Lo usiamo per filtrare il DataFrame e tenere solo le cucine che superano la soglia.

In [ ]:
rows_before = recipes.shape[0]
print("Number of rows of original dataframe is {}.".format(rows_before))

recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

rows_after = recipes.shape[0]
print("Number of rows of processed dataframe is {}.".format(rows_after))
print("{} rows removed!".format(rows_before - rows_after))

### 2.5 Convertire "Yes"/"No" in 1/0

Quasi tutte le colonne di ingredienti sono codificate come testo "Yes"/"No". Per poterle sommare, mediare e usarle in un modello, le convertiamo in numeri con `.replace()`, che sostituisce un valore con un altro in tutto il DataFrame.

In [ ]:
recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

recipes.head()

## 3. Mettendo alla prova la domanda: rice + soy + wasabi + seaweed = giapponese?

Con i dati già puliti e numerici, filtriamo le ricette che combinano quattro ingredienti tipicamente associati alla cucina giapponese.

In [ ]:
check_recipes = recipes.loc[
    (recipes["rice"] == 1) &
    (recipes["soy_sauce"] == 1) &
    (recipes["wasabi"] == 1) &
    (recipes["seaweed"] == 1)
]
check_recipes

**Risposta: no.** Insieme a ricette giapponesi appaiono ricette etichettate come "asian" e "east_asian" con esattamente la stessa combinazione di ingredienti. Un insieme di ingredienti condiviso non basta per identificare una cucina con certezza — serve più informazione, o un modello capace di soppesare molti ingredienti contemporaneamente.

## 4. Ingrediente più popolare: attenzione al bias

Sommiamo ogni colonna di ingredienti (ora che sono 1 e 0, sommare una colonna conta quante volte compare quell'ingrediente nel dataset).

In [ ]:
ing = recipes.iloc[:, 1:].sum(axis=0)  # somma ogni colonna, saltando la colonna 0 ("cuisine")
ing

In [ ]:
ingredient = pd.Series(ing.index.values, index=np.arange(len(ing)))
count = pd.Series(list(ing), index=np.arange(len(ing)))
ing_df = pd.DataFrame(dict(ingredient=ingredient, count=count))
ing_df = ing_df[["ingredient", "count"]]
print(ing_df.to_string())

In [ ]:
ing_df.sort_values(["count"], ascending=False, inplace=True)
ing_df.reset_index(inplace=True, drop=True)
print(ing_df)

I tre ingredienti più frequenti sono **egg** (20.997), **wheat** (20.757) e **butter** (20.699). Ma prima di trarre conclusioni: il dataset contiene circa 40.000 ricette americane contro un numero molto minore del resto delle cucine. Questa classifica è **distorta verso ingredienti tipici della cucina americana**, semplicemente perché ci sono moltissime più ricette di quell'origine. Una scoperta statistica non è mai completa senza chiederti da dove viene il campione.

## 5. Profilo di ingredienti per cucina

Per evitare il bias del conteggio a crudo, calcoliamo la **media per cucina** con `.groupby()` (raggruppa le righe in base al valore di una colonna) seguito da `.mean()` (media di ogni colonna numerica dentro ogni gruppo). Poiché ogni ingrediente è 0 o 1, la sua media dentro una cucina è direttamente la percentuale di ricette di quella cucina che lo includono.

In [ ]:
cuisines = recipes.groupby("cuisine").mean()
cuisines.head()

Ad esempio: la mandorla (`almond`) appare nel 15,65% delle ricette africane, e il burro (`butter`) appare nel 38,11% delle ricette canadesi.

Ora automatizziamo questo: per ogni cucina, mostrare i suoi 4 ingredienti più caratteristici.

In [ ]:
num_ingredients = 4  # quanti ingredienti top mostrare per cucina

def print_top_ingredients(row):
    print(row.name.upper())
    row_sorted = row.sort_values(ascending=False) * 100
    top_ingredients = list(row_sorted.index.values)[0:num_ingredients]
    row_sorted = list(row_sorted)[0:num_ingredients]
    for ind, ingredient in enumerate(top_ingredients):
        print("%s (%d%%)" % (ingredient, row_sorted[ind]), end=' ')
    print("\n")

# .apply(..., axis=1) esegue print_top_ingredients() una volta per ogni riga (ogni cucina)
create_cuisines_profiles = cuisines.apply(print_top_ingredients, axis=1)

Questo è feature engineering in azione: non inventiamo dati dal nulla, usiamo ciò che già sapevamo sul dataset (che ogni colonna è un ingrediente binario) per costruire una vista nuova — il profilo degli ingredienti caratteristici di ogni cucina — che non esisteva come colonna nel CSV originale, ma che è molto più utile per comprendere il problema e, nella prossima lezione, per modellarlo.

## 6. Il caso CHF: Data Preparation completa

*(Caso narrativo, senza codice.)*

Nel caso CHF, Data Preparation è iniziata definendo con precisione cosa contava come "insufficienza cardiaca congestizia", usando **codici di gruppi correlati per diagnosi** e guida clinica di esperti medici — perché la CHF è solo uno dei diversi tipi di insufficienza cardiaca. Poi si è definito il criterio di **ricovero ripetuto**: una finestra di **30 giorni** dopo la dimissione dall'ammissione iniziale (chiamata ammissione indice), basata sull'esperienza clinica.

I record originali erano in **formato transazionale** — più record per paziente, con rimborsi di medici, laboratori, ospedali e servizi clinici; un solo paziente poteva avere centinaia o migliaia di questi record. Il team ha **aggregato** tutto questo a **livello di paziente**, generando nuove colonne (frequenza e data delle visite recenti, diagnosi, procedure, prescrizioni) e considerando **comorbilità** come diabete o ipertensione. Tutto è stato combinato con dati demografici (età, genere, tipo di assicurazione) per produrre **una tabella con un record per paziente**, con una variabile obiettivo binaria: ricovero ripetuto per CHF entro 30 giorni, sì o no.

Il risultato finale: una **coorte di 2.343 pazienti**, divisa in un insieme di addestramento e uno di test, pronta per costruire e validare il modello — esattamente ciò che farai nella prossima lezione, sia nel caso narrativo sia con il dataset di ricette già pulito che hai lasciato pronto in questo notebook.

## Riepilogo della lezione

- Hai applicato statistica descrittiva (univariate, espressioni regolari per verificare colonne) per capire `recipes`.
- Hai visto perché i valori assenti o non validi (come un "999" travestito da età valida) hanno bisogno di interpretazione, non solo di rilevamento.
- Hai pulito `recipes` dall'inizio alla fine: nome della colonna, minuscolo, nomi duplicati di cucina, cucine con poche ricette, e "Yes"/"No" in 1/0.
- Hai scoperto che rice+soy+wasabi+seaweed non identifica in modo univoco la cucina giapponese, e che l'ingrediente "più popolare" del dataset è distorto verso la cucina americana.
- Hai costruito un profilo di ingredienti per cucina con `.groupby().mean()` — un esempio reale di feature engineering.
- Hai percorso il caso CHF di Data Understanding (la definizione di CHF che è dovuta essere ampliata) e Data Preparation (definizione clinica, finestra di 30 giorni, aggregazione a livello di paziente, coorte finale di 2.343 pazienti).

Il DataFrame `recipes` e `cuisines` che hai lasciato in questo notebook sono esattamente quelli che userai nel notebook di pratica di questa lezione, e nella Lezione 5 per costruire il tuo primo albero decisionale.